# v1.3 / 9 两阶段真实图片验证

只使用验证集，输入为真实1280×720完整图像。定位后裁剪并按面积平均缩到64×64，再执行分类器。单帧离线诊断，不代表跨帧运动、视频时延或开发板结果。

In [1]:
# v1.3: project-only Jupyter execution; no board programming.
import os,sys,json,hashlib
from pathlib import Path
import numpy as np
import tensorflow as tf
ROOT=Path.cwd()
while not (ROOT/'AGENTS.md').exists():
    if ROOT==ROOT.parent: raise RuntimeError('Open this notebook inside the CNN project')
    ROOT=ROOT.parent
assert ROOT.name=='fpga-w.-codex-cnn'
sys.path.insert(0,str(ROOT/'ml/tinyml'))
from workflow import export_int8,predict_tflite,digest
os.environ['KERAS_HOME']=str(ROOT/'ml/tinyml/keras')
tf.keras.utils.set_random_seed(1301)
tf.config.threading.set_intra_op_parallelism_threads(6)
tf.config.threading.set_inter_op_parallelism_threads(2)
print('TensorFlow',tf.__version__,'GPU',tf.config.list_physical_devices('GPU'))


TensorFlow 2.16.2 GPU []


In [2]:
from PIL import Image
sys.path.insert(0,str(ROOT/'ml/detector'))
from export import select,iou
from evaluate_pipeline import crop_pixels
labels=['fist','peace','palm','ok','like','no_gesture']
rows=[r for r in json.loads((ROOT/'ml/detector/data/manifest.json').read_text()) if r['split']=='val']
full=[np.asarray(Image.open(ROOT/'ml/detector/data'/r['path'])) for r in rows]
low=np.stack([np.asarray(Image.fromarray(im).resize((256,144),Image.Resampling.BOX)) for im in full])[...,None]
detpath=ROOT/'ml/tinyml/artifacts/detector_conv/hand_detector_int8.tflite'
cls_path=ROOT/'ml/tinyml/artifacts/classifier/gesture_classifier_int8.tflite'
outputs=predict_tflite(detpath,(low.astype(np.float32)-128)/128)
head=np.concatenate([next(x for x in outputs if x.shape[-1]==1),next(x for x in outputs if x.shape[-1]==4)],axis=-1)
pred=[select(np.floor(x*256+.5).astype(np.int32)) for x in head]
crops=np.stack([crop_pixels(im,p['roi']) for im,p in zip(full,pred)])
logits=predict_tflite(cls_path,(crops.astype(np.float32)-128)/128)[0]
cids=np.argmax(logits,axis=1);sorted_logits=np.sort(logits,axis=1);margins=sorted_logits[:,-1]-sorted_logits[:,-2]
results=[]
for threshold in [-512,-384,-256,-128,0,128,256,512]:
    accepted=correct_class=correct_iou50=0
    for row,p,cid,margin in zip(rows,pred,cids,margins):
        if not p['valid'] or p['score']<threshold or cid>=5 or margin<1.5:continue
        accepted+=1
        match=max(row['boxes'],key=lambda b:iou(p['box'],b))
        ok=labels[int(cid)]==match[4]
        correct_class+=ok;correct_iou50+=ok and iou(p['box'],match)>=.5
    results.append(dict(detector_threshold_q8=threshold,frames=len(rows),accepted=accepted,correct_class=correct_class,correct_class_and_iou50=correct_iou50,strict_precision=correct_iou50/max(accepted,1),strict_frame_recall=correct_iou50/len(rows)))
report=dict(version='v1.3',split='val',detector_sha256=digest(detpath),classifier_sha256=digest(cls_path),class_margin=1.5,results=results,hardware_verified=False,note='Real single frames; no empty-scene full-frame set, temporal confirmation or board timing.')
out=ROOT/'ml/tinyml/artifacts/pipeline';out.mkdir(parents=True,exist_ok=True)
(out/'validation_report.json').write_text(json.dumps(report,indent=2))
print(json.dumps(report,indent=2))

{
  "version": "v1.3",
  "split": "val",
  "detector_sha256": "27e969f1614a9875d6d95ebfb4664422ba8de80f6c1f44afec0ea33938d21019",
  "classifier_sha256": "6da37335b4883c2a7d741df91a6740a2267302fac367324e0d987b789a7ccf56",
  "class_margin": 1.5,
  "results": [
    {
      "detector_threshold_q8": -512,
      "frames": 432,
      "accepted": 129,
      "correct_class": 109,
      "correct_class_and_iou50": 95,
      "strict_precision": 0.7364341085271318,
      "strict_frame_recall": 0.2199074074074074
    },
    {
      "detector_threshold_q8": -384,
      "frames": 432,
      "accepted": 129,
      "correct_class": 109,
      "correct_class_and_iou50": 95,
      "strict_precision": 0.7364341085271318,
      "strict_frame_recall": 0.2199074074074074
    },
    {
      "detector_threshold_q8": -256,
      "frames": 432,
      "accepted": 129,
      "correct_class": 109,
      "correct_class_and_iou50": 95,
      "strict_precision": 0.7364341085271318,
      "strict_frame_recall": 0.219907